# CH1 — ChatGPT와 Agent는 도대체 뭐가 다른가?

## CASE: 신규 AI 회의록 서비스 경쟁사 분석

같은 업무를 3가지 방식으로 수행합니다.

1. **Single LLM** — 한 번의 Prompt
2. **Workflow** — 미리 정해진 순서대로 검색 → 요약 → 보고서
3. **Agent** — 현재 정보가 충분한지 판단하고 다음 행동을 스스로 선택

---
### 오늘 가져갈 한 문장
> **Agent란 목표를 받고 다음 행동을 스스로 결정하는 AI다.**

## 0. 공통 환경 준비

외부 웹 검색 대신 `competitor_db.json`을 검색 가능한 데이터베이스처럼 사용합니다.

이유:
- 학생별 검색 결과 차이를 줄일 수 있음
- 검색 API Key가 추가로 필요 없음
- 세 방식의 **구조 차이 자체**에 집중 가능

In [ ]:
from pathlib import Path
import os, json, re, textwrap
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
(PROJECT_ROOT / "outputs").mkdir(exist_ok=True)
load_dotenv(PROJECT_ROOT / ".env")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
client = OpenAI()

brief = (PROJECT_ROOT / "data" / "company_brief.md").read_text(encoding="utf-8")
competitor_db = json.loads(
    (PROJECT_ROOT / "data" / "competitor_db.json").read_text(encoding="utf-8")
)

COMPANIES = list(competitor_db.keys())

print("분석 대상:", COMPANIES)
print("\n--- 회사 브리프 ---\n")
print(brief)

# ROUND 1 — Single LLM

## 조건
- 검색 Tool 없음
- 경쟁사 DB 접근 없음
- 회사 브리프와 경쟁사 이름만 전달
- 단 한 번의 LLM 호출

실제 업무에서 흔히 하는:
> "ChatGPT야, 경쟁사 분석 보고서 만들어줘."

를 그대로 재현합니다.

In [ ]:
single_prompt = f'''
너는 사업전략 담당자다.

아래 신규 서비스 브리프를 바탕으로 경쟁사 분석 보고서를 작성하라.

[브리프]
{brief}

[경쟁사]
{", ".join(COMPANIES)}

다음 항목을 포함하라.
- 경쟁사별 타깃
- 가격
- 주요 기능
- 장단점
- 보안/기업 기능
- 우리 서비스의 차별화 기회

중요:
확인할 수 없는 사실은 추측하지 말고 "정보 부족"이라고 표시하라.
'''

single = client.responses.create(model=MODEL, input=single_prompt).output_text
print(single)

(PROJECT_ROOT / "outputs" / "01_single_llm.txt").write_text(single, encoding="utf-8")

### 관찰 포인트

Single LLM은 보고서의 **형식**은 잘 만듭니다.

하지만 지금 모델에게는 경쟁사별 실제 자료가 없습니다.

따라서 정상적인 결과라면:
- 가격 → 정보 부족
- 리뷰 → 정보 부족
- 보안 → 정보 부족
- 장단점 → 구체적 근거 부족

이 나타나야 합니다.

**즉, 생각은 할 수 있지만 조사할 손과 발이 없습니다.**

# ROUND 2 — Fixed Workflow

이번에는 검색 기능을 줍니다.

하지만 AI가 무엇을 검색할지는 결정하지 않습니다.

사람이 미리 다음 순서를 코딩합니다.

```
경쟁사별
  ↓
기본정보 검색
  ↓
가격 검색
  ↓
기능 검색
  ↓
요약
  ↓
최종 보고서
```

일부러 `고객 리뷰`, `보안`은 Workflow에서 제외해두었습니다.

In [ ]:
# 사내 검색 시스템을 흉내낸 Tool
def search_db(company: str, field: str):
    if company not in competitor_db:
        return None
    return competitor_db[company].get(field)

def run_fixed_workflow():
    collected = {}

    for company in COMPANIES:
        collected[company] = {
            "overview": search_db(company, "overview"),
            "pricing": search_db(company, "pricing"),
            "features": search_db(company, "features"),
            # 일부러 reviews / security / target 은 검색하지 않음
        }

    return collected

workflow_data = run_fixed_workflow()
workflow_data

In [ ]:
workflow_prompt = f'''
너는 사업전략 담당자다.

아래 회사 브리프와 고정 Workflow가 조사한 경쟁사 자료를 이용해
경쟁사 분석 보고서를 작성하라.

[브리프]
{brief}

[수집 자료]
{json.dumps(workflow_data, ensure_ascii=False, indent=2)}

다음 항목을 포함하라.
- 경쟁사별 타깃
- 가격
- 주요 기능
- 고객 리뷰에서 발견되는 장단점
- 보안/기업 기능
- 우리 서비스의 차별화 기회

자료에 없는 내용은 추측하지 말고 "정보 부족"이라고 표시하라.
'''

workflow_report = client.responses.create(
    model=MODEL,
    input=workflow_prompt
).output_text

print(workflow_report)

(PROJECT_ROOT / "outputs" / "02_workflow.txt").write_text(
    workflow_report, encoding="utf-8"
)

### 관찰 포인트

Single LLM보다 훨씬 좋아졌습니다.

왜냐하면 이제 **실제 자료를 검색했기 때문**입니다.

하지만 Workflow는 사람이 정한 것만 합니다.

```
overview
pricing
features
```

만 검색하도록 만들었기 때문에,

보고서에서 중요한:
- 실제 고객 리뷰
- 보안/기업 기능
- 명확한 타깃

이 여전히 부족합니다.

> Workflow는 **정해진 일을 잘한다.**

# ROUND 3 — Agent

이제 AI에게 다음 Tool을 줍니다.

```
search_overview
search_pricing
search_features
search_reviews
search_security
search_target
```

그러나 **어떤 순서로 몇 번 검색할지는 정해주지 않습니다.**

Agent가 현재 확보된 자료를 보고:

> "최종 보고서를 만들기에 무엇이 부족하지?"

를 판단해서 다음 행동을 고릅니다.

In [ ]:
AVAILABLE_FIELDS = [
    "overview",
    "pricing",
    "features",
    "reviews",
    "security",
    "target",
    "weakness",
]

FIELD_DESCRIPTIONS = {
    "overview": "서비스 개요",
    "pricing": "가격/요금제",
    "features": "주요 기능",
    "reviews": "실제 고객 리뷰에서 반복되는 평가",
    "security": "보안/기업 기능",
    "target": "주요 타깃 고객",
    "weakness": "대표적인 약점"
}

def tool_search(company, field):
    print(f"🔎 TOOL CALL → {company} / {field}")
    return search_db(company, field)

## Agent의 의사결정 형식

매 Step마다 Agent는 아래 둘 중 하나만 선택합니다.

```json
{
  "action": "search",
  "company": "MeetNote AI",
  "field": "reviews",
  "reason": "장단점 판단을 위한 고객 근거가 아직 없다"
}
```

또는

```json
{
  "action": "finish",
  "reason": "모든 핵심 평가 항목의 근거가 충분하다"
}
```

여기서 Workflow와 가장 큰 차이가 발생합니다.

**다음 action이 코드에 고정돼 있지 않습니다.**

In [ ]:
def extract_json(text):
    # 모델이 ```json ... ```로 감싸더라도 처리
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"JSON을 찾지 못했습니다: {text}")
    return json.loads(match.group(0))

def agent_decide(collected, step):
    prompt = f"""
    너는 경쟁사 리서치 Agent다.

    목표:
    아래 브리프에 맞는 경쟁사 분석 보고서를 만들기 위해
    필요한 근거를 충분히 조사하라.

    [브리프]
    {brief}

    [분석 대상]
    {COMPANIES}

    [현재까지 수집된 정보]
    {json.dumps(collected, ensure_ascii=False, indent=2)}

    [사용 가능한 field]
    {json.dumps(FIELD_DESCRIPTIONS, ensure_ascii=False, indent=2)}

    현재 Step: {step}

    중요한 완료 기준:
    모든 경쟁사에 대해 최소한 다음 정보가 있어야 한다.
    - overview
    - pricing
    - features
    - reviews
    - security
    - target

    현재 자료를 보고 다음 행동을 정확히 하나 선택하라.

    1) 추가 검색이 필요하면:
    {{
      "action": "search",
      "company": "정확한 회사명",
      "field": "정확한 field명",
      "reason": "왜 이 정보가 다음으로 필요한지"
    }}

    2) 완료 기준을 만족하면:
    {{
      "action": "finish",
      "reason": "왜 충분한지"
    }}

    이미 수집한 company/field 조합은 다시 검색하지 마라.
    JSON만 출력하라.
    """

    result = client.responses.create(model=MODEL, input=prompt).output_text
    return extract_json(result)

In [ ]:
def run_agent(max_steps=30):
    collected = {company: {} for company in COMPANIES}
    trajectory = []

    for step in range(1, max_steps + 1):
        decision = agent_decide(collected, step)

        print(f"\nSTEP {step}")
        print("🤖 DECISION:", decision)

        trajectory.append({
            "step": step,
            **decision
        })

        if decision["action"] == "finish":
            print("\n✅ Agent가 조사를 종료했습니다.")
            break

        company = decision["company"]
        field = decision["field"]

        if company not in COMPANIES or field not in AVAILABLE_FIELDS:
            print("⚠️ 잘못된 action. 다음 step으로 넘어갑니다.")
            continue

        if field in collected[company]:
            print("⚠️ 이미 조사한 항목입니다. 다음 step으로 넘어갑니다.")
            continue

        collected[company][field] = tool_search(company, field)

    return collected, trajectory

agent_data, trajectory = run_agent()

In [ ]:
pd.DataFrame(trajectory)

## Agent가 실제로 무엇을 했는지 확인

위 표가 핵심입니다.

Workflow에서는:

```
Step 1 = overview
Step 2 = pricing
Step 3 = features
```

가 코드에 미리 정해져 있었습니다.

Agent에서는 매 Step의 다음 행동을 LLM이 결정합니다.

실제 실행할 때:
- 어느 회사를 먼저 볼지
- 어떤 정보를 먼저 검색할지
- 언제 그만둘지

가 달라질 수 있습니다.

In [ ]:
agent_prompt = f'''
너는 사업전략 담당자다.

아래 브리프와 Agent가 스스로 수집한 자료를 바탕으로
의사결정에 바로 사용할 수 있는 경쟁사 분석 보고서를 작성하라.

[브리프]
{brief}

[Agent가 수집한 근거]
{json.dumps(agent_data, ensure_ascii=False, indent=2)}

반드시 포함:
1. 경쟁사별 타깃
2. 가격 비교
3. 주요 기능 비교
4. 고객 리뷰 기반 장단점
5. 보안/기업 기능
6. 한국 중소기업 시장에서의 빈틈
7. 우리 서비스가 가져갈 차별화 전략 3가지

각 판단에는 어떤 근거에서 나온 것인지 구체적으로 연결하라.
'''

agent_report = client.responses.create(
    model=MODEL,
    input=agent_prompt
).output_text

print(agent_report)

(PROJECT_ROOT / "outputs" / "03_agent.txt").write_text(
    agent_report, encoding="utf-8"
)

# 세 결과를 같은 기준으로 채점하기

점수를 손으로 적어 넣지 않습니다.
**세 방식이 만든 최종 보고서 텍스트**를 똑같은 정답지로 채점합니다.

정답지(`FACT_CHECKLIST`)는 `competitor_db.json`에만 있는 사실로 만들었습니다.
(예: MeetNote AI 가격 "19달러", 보안 "SOC 2")
모델이 미리 알 수 없는 값이기 때문에, 보고서에 이 값이 나왔다면 **근거를 보고 썼다**는 뜻입니다.

- 회사 3곳 × 항목 5개(pricing / features / reviews / security / target) = 15개 사실
- 보고서에 키워드가 하나라도 있으면 맞힌 것으로 셉니다

주의:
- 이 점수는 '글의 품질' 점수가 아닙니다. **보고서가 실제 근거를 얼마나 담았는가**를 보여줍니다.
- 키워드 매칭이라 다른 말로 바꿔 쓴 표현은 놓칠 수 있습니다. 더 정교하게 하려면 LLM에게 채점을 맡기는 **LLM-as-judge** 방식을 씁니다.
- 아래 상세표에서 어떤 칸을 맞혔는지 직접 확인해 보세요. Single LLM이 맞힌 칸이 있다면, 근거를 보고 쓴 걸까요, 우연히 맞힌 걸까요?

In [ ]:
import re

# 정답지: competitor_db.json에만 있는 사실
# 키워드 중 하나라도 보고서에 나오면 '맞힘'
FACT_CHECKLIST = {
    "MeetNote AI": {
        "pricing":  ["19달러", "$19"],
        "features": ["CRM"],
        "reviews":  ["고유명사"],
        "security": ["SOC2"],
        "target":   ["20~200", "세일즈"],
    },
    "MinutesFlow": {
        "pricing":  ["15달러", "$15"],
        "features": ["Jira"],
        "reviews":  ["초기설정"],
        "security": ["SSO", "데이터리전"],
        "target":   ["중견"],
    },
    "Briefly AI": {
        "pricing":  ["12달러", "$12"],
        "features": ["이메일초안", "메일초안"],
        "reviews":  ["관리자기능이부족", "관리자기능부족"],
        "security": ["기본암호화"],
        "target":   ["1인사업자", "프리랜서"],
    },
}

def norm(text):
    return re.sub(r"\s+", "", text)   # 띄어쓰기 차이 무시

def fact_check(report):
    t = norm(report)
    rows = []
    for company, facts in FACT_CHECKLIST.items():
        for field, keywords in facts.items():
            rows.append({
                "company": company,
                "field": field,
                "hit": any(norm(k) in t for k in keywords),
            })
    return pd.DataFrame(rows)

reports = {
    "Single LLM": single,
    "Fixed Workflow": workflow_report,
    "Agent": agent_report,
}
results = {name: fact_check(report) for name, report in reports.items()}

comparison = pd.DataFrame([
    {
        "방식": name,
        "맞힌 사실": int(df.hit.sum()),
        "전체 사실": len(df),
        "정확도 (%)": round(df.hit.mean() * 100, 1),
    }
    for name, df in results.items()
])

# 어떤 칸을 맞혔는지 직접 확인
detail = results["Single LLM"][["company", "field"]].copy()
for name, df in results.items():
    detail[name] = df.hit.map({True: "O", False: "-"})

display(comparison)
display(detail)

In [ ]:
import matplotlib.pyplot as plt

ax = comparison.plot(
    kind="bar",
    x="방식",
    y="정확도 (%)",
    legend=False,
    ylim=(0, 105),
    rot=0,
    title="Fact Check: Single LLM vs Workflow vs Agent"
)
ax.set_xlabel("")
ax.set_ylabel("Facts found in report (%)")
plt.show()

# 수업 토론

## Q1. Workflow에 reviews와 security 검색을 추가하면 Agent가 필요 없지 않나요?

맞습니다.

**업무의 단계와 필요한 정보가 항상 고정돼 있다면 Workflow가 더 좋은 선택일 수 있습니다.**

Agent가 필요한 이유는 모든 문제에 Agent를 써야 해서가 아닙니다.

다음 행동이 상황마다 달라질 때 가치가 생깁니다.

---

## Q2. 그러면 Agent의 본질은?

```
Workflow

사람이:
다음에는 A
그다음 B
그다음 C
라고 결정
```

vs.

```
Agent

목표를 줌
  ↓
현재 상태 확인
  ↓
다음 행동을 AI가 결정
  ↓
결과 관찰
  ↓
다음 행동 다시 결정
```

---

# 오늘의 한 문장

> ## Agent란 목표를 받고 다음 행동을 스스로 결정하는 AI다.

# 과제

아래 중 하나를 골라 같은 구조로 바꿔보세요.

1. 채용 후보자 리서치
2. 신규 거래처 사전 조사
3. 마케팅 캠페인 경쟁사 조사
4. 공급업체 비교
5. 기업 교육 제안 전 고객사 조사

단, 바로 Agent부터 만들지 마세요.

**Single LLM → Fixed Workflow → Agent**

세 방식 모두 만들어보고 무엇이 다른지 설명하세요.